
# Interaction and steric descriptors for Tg

*Author: Suhmi. Builds on notebooks 06, 08 and 10.*

## Purpose

Most of our Tg-related descriptors describe **chain stiffness** (rings, rotatable bonds),
and they largely repeat each other. Tg also depends on factors that were only roughly covered:

| Factor | Effect on Tg | New descriptors |
|---|---|---|
| Hydrogen bonds | stronger → higher | `HDonorsPerBackboneAtom`, `BackboneHDonorsPerAtom`, `HBondGroupsPerBackboneAtom` |
| Dipoles (C=O, C≡N, C–Cl, S=O, ...) | stronger → higher | `PolarGroupsPerHeavyAtom`, `PolarGroupsPerBackboneAtom` |
| Steric hindrance at the backbone (e.g. PMA → PMMA) | bulkier → higher | `BackboneSubstitutedFraction`, `BackboneDisubstitutedAtoms`, `StericBulkPerBackboneAtom` |
| Symmetric disubstitution (e.g. PVC → PVDC) | symmetric → lower | `SymmetricDisubstitution` |
| Internal plasticisation by flexible side chains | longer → lower | `SideChainAtomsPerBackboneAtom`, `FlexibleSideBondsPerBackboneAtom` |

The values are given **per backbone atom**, so they describe the density along the chain.
All 11 descriptors are calculated from the SMILES (`src/polymer_features.py`,
`calculate_interaction_descriptors`).

## Rules

The same splits and folds as before; cross-validation only. **The validation subsets are frozen.**
The external Tg data is the cleaned set from notebook 10 (`data/processed/external_tg_clean.csv`),
from which all Tg validation polymers were removed.


## 1. Setup

In [1]:

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from rdkit import Chem
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold, train_test_split
from sklearn.pipeline import make_pipeline
from xgboost import XGBRegressor

# Make the project's src package importable from the notebooks folder
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.polymer_features import (
    INTERACTION_COLUMNS, SHAPE_COLUMNS, build_chain_features, build_features,
    build_interaction_features, calculate_interaction_descriptors,
)

RANDOM_STATE = 42
VALIDATION_FRACTION = 0.20
N_FOLDS = 5

target_columns = ["Tg", "FFV", "Tc", "Density", "Rg"]


## 2. Check on known polymers

In [2]:

examples = {
    "Polyethylene": "*CC*", "Polypropylene": "*CC(*)C", "Polyisobutylene": "*CC(*)(C)C",
    "PVC": "*CC(*)Cl", "PVDC": "*CC(*)(Cl)Cl", "PMA": "*CC(*)C(=O)OC", "PMMA": "*CC(*)(C)C(=O)OC",
    "Poly(hexyl methacrylate)": "*CC(*)(C)C(=O)OCCCCCC", "Polystyrene": "*CC(*)c1ccccc1",
    "Nylon 6": "*CCCCCC(=O)N*", "PVA": "*CC(*)O", "PAN": "*CC(*)C#N",
}
display(pd.DataFrame({name: calculate_interaction_descriptors(Chem.MolFromSmiles(s))
                      for name, s in examples.items()}).loc[INTERACTION_COLUMNS].round(2))


,Polyethylene,Polypropylene,Polyisobutylene,PVC,PVDC,PMA,PMMA,Poly(hexyl methacrylate),Polystyrene,Nylon 6,PVA,PAN
BackboneSubstitutedFraction,0.0,0.5,0.5,0.50,0.5,0.50,0.50,0.50,0.5,0.14,0.5,0.50
BackboneDisubstitutedAtoms,0.0,0.0,1.0,0.00,1.0,0.00,1.00,1.00,0.0,0.00,0.0,0.00
StericBulkPerBackboneAtom,0.0,0.5,1.0,0.50,1.0,1.50,2.00,2.00,1.5,0.14,0.5,1.00
SymmetricDisubstitution,0.0,0.0,1.0,0.00,1.0,0.00,0.00,0.00,0.0,0.00,0.0,0.00
HDonorsPerBackboneAtom,0.0,0.0,0.0,0.00,0.0,0.00,0.00,0.00,0.0,0.14,0.5,0.00
BackboneHDonorsPerAtom,0.0,0.0,0.0,0.00,0.0,0.00,0.00,0.00,0.0,0.14,0.0,0.00
HBondGroupsPerBackboneAtom,0.0,0.0,0.0,0.00,0.0,0.00,0.00,0.00,0.0,0.14,0.0,0.00
PolarGroupsPerHeavyAtom,0.0,0.0,0.0,0.33,0.5,0.17,0.14,0.08,0.0,0.12,0.0,0.25
PolarGroupsPerBackboneAtom,0.0,0.0,0.0,0.50,1.0,0.50,0.50,0.50,0.0,0.14,0.0,0.50
SideChainAtomsPerBackboneAtom,0.0,0.5,1.0,0.50,1.0,2.00,2.50,5.00,3.0,0.14,0.5,1.00


## 3. Features, splits and folds

In [3]:

train = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "train.csv")
external = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "external_tg_clean.csv")


def feature_table(smiles, shape):
    bplus, fingerprints = build_features(smiles)
    chain = build_chain_features(smiles)
    interaction = build_interaction_features(smiles)
    return pd.concat([bplus, chain, shape, interaction, fingerprints], axis=1)


shape_ours = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "shape_descriptors.csv",
                         index_col=0)[SHAPE_COLUMNS]
features_ours = feature_table(train["SMILES"], shape_ours)
features_external = feature_table(external["SMILES"], external[SHAPE_COLUMNS])
y_external = external["Tg"].to_numpy()

new_columns = INTERACTION_COLUMNS
fingerprint_columns = [c for c in features_ours.columns if c.startswith("Morgan_")]
old_columns = [c for c in features_ours.columns if c not in new_columns + fingerprint_columns]
feature_sets = {
    "73 descriptors + D": old_columns + fingerprint_columns,
    "84 descriptors + D (+ interaction)": old_columns + new_columns + fingerprint_columns,
}
for name, columns in feature_sets.items():
    print(f"{name}: {len(columns)} features")

cv_folds = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
labels = train[target_columns].to_numpy(dtype=float)
train_rows, fold_rows = {}, {}
for target in target_columns:
    labelled = train.index[train[target].notna()]
    train_index, _ = train_test_split(
        labelled, test_size=VALIDATION_FRACTION, random_state=RANDOM_STATE, shuffle=True
    )
    train_rows[target] = np.asarray(train_index)
    fold_rows[target] = [train_rows[target][test_idx] for _, test_idx in cv_folds.split(train_index)]
expected_sizes = {"Tg": 408, "FFV": 5624, "Tc": 589, "Density": 490, "Rg": 491}
assert {t: len(rows) for t, rows in train_rows.items()} == expected_sizes

# Correlation of the new descriptors with Tg (our Tg training polymers only)
tg_rows = train_rows["Tg"]
display(features_ours.loc[tg_rows, new_columns]
        .corrwith(train.loc[tg_rows, "Tg"], method="spearman").round(2)
        .rename("Spearman correlation with Tg").to_frame())


73 descriptors + D: 1097 features
84 descriptors + D (+ interaction): 1108 features


,Spearman correlation with Tg
BackboneSubstitutedFraction,0.07
BackboneDisubstitutedAtoms,0.02
StericBulkPerBackboneAtom,0.01
SymmetricDisubstitution,0.07
HDonorsPerBackboneAtom,0.06
BackboneHDonorsPerAtom,0.10
HBondGroupsPerBackboneAtom,0.13
PolarGroupsPerHeavyAtom,-0.04
PolarGroupsPerBackboneAtom,-0.03
SideChainAtomsPerBackboneAtom,-0.03



## 4. Tg: with and without the new descriptors and the external data

The two best set-ups so far: tuned XGBoost on our data only, and default XGBoost with
the external data and a source flag (notebook 10).


In [4]:

tg_params = dict(n_estimators=750, learning_rate=0.013339162948869679, max_depth=2,
                 min_child_weight=1, subsample=0.6550083776583973,
                 colsample_bytree=0.7394663949166917, reg_lambda=7.472397689332939)
default_params = dict(n_estimators=500, learning_rate=0.05, max_depth=4, subsample=0.8,
                      colsample_bytree=0.8)
TG = target_columns.index("Tg")


def cross_validate(target, columns, params, with_external=False):
    t = target_columns.index(target)
    X = features_ours[columns].to_numpy(dtype=float)
    if with_external:
        X = np.hstack([X, np.zeros((len(X), 1))])
        X_ext = np.hstack([features_external[columns].to_numpy(dtype=float),
                           np.ones((len(features_external), 1))])
    rows_out = []
    for fold in range(N_FOLDS):
        rows = fold_rows[target][fold]
        fit_rows = np.setdiff1d(train_rows[target], rows)
        X_fit, y_fit = X[fit_rows], labels[fit_rows, t]
        if with_external:
            X_fit, y_fit = np.vstack([X_fit, X_ext]), np.concatenate([y_fit, y_external])
        model = make_pipeline(SimpleImputer(strategy="median"),
                              XGBRegressor(**params, random_state=RANDOM_STATE, n_jobs=-1))
        y_pred = model.fit(X_fit, y_fit).predict(X[rows])
        y_true = labels[rows, t]
        baseline = np.mean(np.abs(y_true - np.median(labels[fit_rows, t])))
        mae = mean_absolute_error(y_true, y_pred)
        rows_out.append({"fold": fold, "MAE": mae, "error_reduction": 100 * (1 - mae / baseline),
                         "R2": r2_score(y_true, y_pred)})
    return rows_out


setups = {
    "Tuned XGBoost, our data": (tg_params, False),
    "Default XGBoost, our data": (default_params, False),
    "Default XGBoost, + external + source flag": (default_params, True),
}
tg_results = []
for set_name, columns in feature_sets.items():
    for setup, (params, with_external) in setups.items():
        for row in cross_validate("Tg", columns, params, with_external):
            tg_results.append({"features": set_name, "setup": setup, **row})

tg_df = pd.DataFrame(tg_results)
tg_summary = tg_df.groupby(["setup", "features"]).agg(
    MAE=("MAE", "mean"), error_reduction=("error_reduction", "mean"),
    error_reduction_std=("error_reduction", "std"), R2=("R2", "mean"))
display(tg_summary.round(2))


MAE  \
setup                                     features                                    
Default XGBoost, + external + source flag 73 descriptors + D                  44.86   
                                          84 descriptors + D (+ interaction)  45.90   
Default XGBoost, our data                 73 descriptors + D                  50.60   
                                          84 descriptors + D (+ interaction)  50.19   
Tuned XGBoost, our data                   73 descriptors + D                  48.16   
                                          84 descriptors + D (+ interaction)  48.42   

                                                                              error_reduction  \
setup                                     features                                              
Default XGBoost, + external + source flag 73 descriptors + D                            49.43   
                                          84 descriptors + D (+ interaction)            48.23   
Default XGBoost, our data                 73 descriptors + D                            42.93   
                                          84 descriptors + D (+ interaction)            43.37   
Tuned XGBoost, our data                   73 descriptors + D                            45.62   
                                          84 descriptors + D (+ interaction)            45.35   

                                                                              error_reduction_std  \
setup                                     features                                                  
Default XGBoost, + external + source flag 73 descriptors + D                                 4.53   
                                          84 descriptors + D (+ interaction)                 2.93   
Default XGBoost, our data                 73 descriptors + D                                 4.24   
                                          84 descriptors + D (+ interaction)                 3.45   
Tuned XGBoost, our data                   73 descriptors + D                                 2.57   
                                          84 descriptors + D (+ interaction)                 2.39   

                                                                                R2  
setup                                     features                                  
Default XGBoost, + external + source flag 73 descriptors + D                  0.72  
                                          84 descriptors + D (+ interaction)  0.71  
Default XGBoost, our data                 73 descriptors + D                  0.66  
                                          84 descriptors + D (+ interaction)  0.66  
Tuned XGBoost, our data                   73 descriptors + D                  0.69  
                                          84 descriptors + D (+ interaction)  0.68

In [5]:

# Which new descriptors does the best model use? (gain importance, top 15)
X_fit = np.vstack([
    np.hstack([features_ours.loc[tg_rows, feature_sets["84 descriptors + D (+ interaction)"]]
               .to_numpy(dtype=float), np.zeros((len(tg_rows), 1))]),
    np.hstack([features_external[feature_sets["84 descriptors + D (+ interaction)"]]
               .to_numpy(dtype=float), np.ones((len(features_external), 1))]),
])
y_fit = np.concatenate([labels[tg_rows, TG], y_external])
model = make_pipeline(SimpleImputer(strategy="median"),
                      XGBRegressor(**default_params, random_state=RANDOM_STATE, n_jobs=-1)).fit(X_fit, y_fit)
names = feature_sets["84 descriptors + D (+ interaction)"] + ["SourceFlag"]
importance = pd.Series(model[-1].feature_importances_, index=names).nlargest(15)
display(importance.rename(lambda n: f"{n} ★" if n in new_columns else n).round(3).to_frame("importance"))


,importance
BackboneRingFraction,0.127
NumRings,0.035
BackboneRotatableFraction,0.032
Morgan_420,0.022
Morgan_887,0.015
Morgan_792,0.015
VdWVolumePerHeavyAtom,0.011
AmideBackbone,0.010
Morgan_771,0.009
Morgan_13,0.009


## 5. The other properties

A quick check with default XGBoost (our data only).

In [6]:

other_rows = []
for target in target_columns[1:]:
    for set_name, columns in feature_sets.items():
        for row in cross_validate(target, columns, default_params):
            other_rows.append({"target": target, "features": set_name, **row})
other_df = pd.DataFrame(other_rows)
other_summary = other_df.pivot_table(index="target", columns="features", values="error_reduction")
other_summary = other_summary.loc[target_columns[1:], list(feature_sets)]
other_summary["gain (points)"] = other_summary.iloc[:, 1] - other_summary.iloc[:, 0]
print("Cross-validated error reduction vs. median baseline (%)")
display(other_summary.round(1))


Cross-validated error reduction vs. median baseline (%)


features,73 descriptors + D,84 descriptors + D (+ interaction),gain (points)
target,,,
FFV,67.6,67.8,0.2
Tc,65.1,65.8,0.7
Density,77.4,76.9,-0.5
Rg,66.0,65.6,-0.4


#### Findings: interaction and steric descriptors

- The descriptors behave as intended on known polymers (PMMA bulkier than PMA, PVDC and PIB
  symmetric, Nylon 6 with backbone H-bond groups, long flexible side chains for poly(hexyl methacrylate)).
- **But they are hardly related to Tg in our data:** Spearman correlations between −0.13 and +0.13
  (for comparison, the stiffness descriptors reach ±0.6).
- **They do not improve the model:** Tg MAE 48.2 → 48.4 °C (our data, tuned XGBoost) and
  44.9 → 45.9 °C (with external data). None of them is among the 15 most important features.
  For the other properties the changes are within ±0.7 points.

**Interpretation.** Across this very diverse set of polymers, Tg is dominated by backbone stiffness.
H-bonding, dipoles, steric hindrance and side-chain plasticisation are real effects, but they matter
mainly *within* a family of similar polymers (e.g. PMA vs. PMMA), and the functional-group descriptors
and fingerprints already capture most of this information.

**Decision:** the interaction descriptors are kept in `src/polymer_features.py` for reference, but not
used in the models. The best Tg set-up remains: 73 descriptors + fingerprints, default XGBoost,
our data + external data with a source flag (MAE 44.9 °C).
